In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv("chotot_raw.csv")

df.head()

,title,price,area,location,description,Diện tích đất:,Giá/m2:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,...,Diện tích:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"2,38 tỷ- 100 m2",- 100 m2,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",Còn lô giá rẻ nhất khu vực: Nam Cẩm Lệ\n✔️ Đườ...,100 m2,"23,8 triệu/m2",Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,18 tỷ- 79 m2,- 79 m2,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","Nhà 1 trệt 2 lầu\nDiện tích 4,15x18,8\n4 phòng...",79 m²,"227,85 triệu/m²",Nam,Đang chờ sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2",1 tỷ- 500 m2,- 500 m2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...","Vài lô liền kề nằm ngay kcn , tthc bầu bàng \n...",500 m2,2 triệu/m2,Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn",525 triệu- 60 m2,- 60 m2,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...","Nhà chính chủ mới xây đường võ văn vân, vĩnh l...",60 m²,"8,75 triệu/m²",Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,440 triệu- 150 m2,- 150 m2,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...","bán lô đất đẹp mặt tiền đường nhựa thôn 2, Suố...",150 m2,"2,93 triệu/m2",Bắc,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 📍 Bước 1 — Xử lý cột `location`

Cột `location` chứa địa chỉ dạng chuỗi tự do, cần được tách thành 4 thành phần chuẩn hóa để phục vụ phân tích theo khu vực.

## 🗺️ Location Parsing — Quy tắc xử lý

### Mục tiêu
Tách chuỗi địa chỉ trong cột `location` thành 4 cột riêng biệt:

| Cột | Ý nghĩa | Ví dụ |
|-----|---------|-------|
| `street` | Tên đường | `đường Lê Lợi` |
| `ward` | Phường / Xã | `phường Bến Nghé` |
| `district` | Quận / Huyện | `quận 1` |
| `city` | Tỉnh / Thành phố | `tp. Hồ Chí Minh` |

---

### 1. Tiền xử lý (Preprocessing)
- Chuyển toàn bộ text về **chữ thường**
- Loại bỏ khoảng trắng dư thừa
- Xóa ký tự nhiễu dạng `||số`
- Tách chuỗi theo dấu phẩy `,`
- Bỏ phần tử đầu nếu là số thuần (ví dụ: `"36"`)

---

### 2. Nhận diện theo prefix (Rule-based)

Mỗi phần tử sau khi split được phân loại dựa trên **từ đứng đầu**:

| Prefix | Gán vào |
|--------|---------|
| `đường` | `street` |
| `phường`, `xã`, `thị trấn`, `thôn`, `kênh` | `ward` |
| `quận`, `huyện`, `thị xã` | `district` |
| `tỉnh`, `tp` | `city` |
| `thành phố` | `district` nếu chưa có, ngược lại `city` |

---

### 3. Fallback Rule
Nếu chưa xác định được `city` và chuỗi có **≥ 3 phần tử** → lấy phần tử **cuối cùng** làm `city`.

---

### 4. Fill các field còn thiếu
Sau khi loại bỏ các phần tử đã dùng, các phần còn lại được gán tuần tự theo thứ tự: `street` → `ward` → `district`.

In [4]:
df[["location"]].head(10)

,location
0,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ..."
1,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh"
2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu..."
3,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch..."
4,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ..."
5,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ..."
6,"Đường số 1, Phường Trường Thọ, Quận Thủ Đức,..."
7,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả..."
8,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả..."
9,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch..."


In [5]:
def clean_location(text):
    if pd.isna(text):
        return None
    
    text = text.lower().strip()
    
    # bỏ ký tự rác
    text = re.sub(r"\|\|\d+", "", text)
    
    # normalize space
    text = re.sub(r"\s+", " ", text)
    
    return text

df["location_clean"] = df["location"].apply(clean_location)

df[["location", "location_clean"]].head(10)

,location,location_clean
0,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...","đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ..."
1,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","đường cao đạt, phường 1, quận 5, tp hồ chí minh"
2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...","đường quốc lộ 13, thị trấn lai uyên, huyện bàu..."
3,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...","đường võ văn vân, xã vĩnh lộc b, huyện bình ch..."
4,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...","thôn 2, xã suối rao, huyện châu đức, bà rịa - ..."
5,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...","đường lý thái tổ, xã đạm bri, thành phố bảo lộ..."
6,"Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...","đường số 1, phường trường thọ, quận thủ đức,..."
7,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...","đường quốc lộ 20, thị trấn lộc thắng, huyện bả..."
8,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...","đường 359, xã tân dương, huyện thuỷ nguyên, hả..."
9,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...","đường phó cơ điều, phường 12, quận 5, tp hồ ch..."


In [6]:
def debug_split(text):
    if pd.isna(text):
        return None
    return [p.strip() for p in text.split(",")]

df["parts"] = df["location_clean"].apply(debug_split)

df[["location_clean", "parts"]].head(10)

,location_clean,parts
0,"đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ...","[đường lỗ giáng 8, phường hòa xuân, quận cẩm l..."
1,"đường cao đạt, phường 1, quận 5, tp hồ chí minh","[đường cao đạt, phường 1, quận 5, tp hồ chí minh]"
2,"đường quốc lộ 13, thị trấn lai uyên, huyện bàu...","[đường quốc lộ 13, thị trấn lai uyên, huyện bà..."
3,"đường võ văn vân, xã vĩnh lộc b, huyện bình ch...","[đường võ văn vân, xã vĩnh lộc b, huyện bình c..."
4,"thôn 2, xã suối rao, huyện châu đức, bà rịa - ...","[thôn 2, xã suối rao, huyện châu đức, bà rịa -..."
5,"đường lý thái tổ, xã đạm bri, thành phố bảo lộ...","[đường lý thái tổ, xã đạm bri, thành phố bảo l..."
6,"đường số 1, phường trường thọ, quận thủ đức,...","[đường số 1, phường trường thọ, quận thủ đức..."
7,"đường quốc lộ 20, thị trấn lộc thắng, huyện bả...","[đường quốc lộ 20, thị trấn lộc thắng, huyện b..."
8,"đường 359, xã tân dương, huyện thuỷ nguyên, hả...","[đường 359, xã tân dương, huyện thuỷ nguyên, h..."
9,"đường phó cơ điều, phường 12, quận 5, tp hồ ch...","[đường phó cơ điều, phường 12, quận 5, tp hồ c..."


In [7]:
import unicodedata
import re

def normalize_text(text):
    text = unicodedata.normalize('NFC', text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [8]:
def starts_with_any(text, keywords):
    return any(text.startswith(k + " ") or text == k for k in keywords)


def parse_location(text):
    if pd.isna(text):
        return pd.Series({
            "street": None,
            "ward": None,
            "district": None,
            "city": None
        })
    
    # 🔥 normalize
    text = normalize_text(text)
    
    # 🔹 split
    parts = [p.strip() for p in text.split(",") if p.strip() != ""]
    
    # 🔹 remove số đầu (vd: "36")
    if len(parts) > 1 and re.fullmatch(r"\d+", parts[0]):
        parts = parts[1:]
        # print("PARTS:", parts)  # debug nếu cần
    
    result = {
        "street": None,
        "ward": None,
        "district": None,
        "city": None
    }
    
    unknown_parts = []
    
    # 🔥 1. Keyword detection (prefix-based)
    for part in parts:
        p = part.strip()
        
        if starts_with_any(p, ["phường", "xã", "thị trấn", "thôn", "kênh"]):
            result["ward"] = part
        
        elif starts_with_any(p, ["quận", "huyện", "thị xã"]):
            result["district"] = part
        
        elif starts_with_any(p, ["đường"]):
            result["street"] = part
        
        elif starts_with_any(p, ["tỉnh", "tp"]):
            result["city"] = part
        
        elif p.startswith("thành phố"):
            if result["district"] is None:
                result["district"] = part
            else:
                result["city"] = part
        
        else:
            unknown_parts.append(part)   # ✅ FIX INDENT
    
    # 🔥 2. Fix city (fallback)
    if result["city"] is None and len(parts) >= 3:
        result["city"] = parts[-1]
    
    # 🔥 3. Remove các phần đã dùng
    used = set([v for v in result.values() if v is not None])
    remaining = [p for p in parts if p not in used]
    
    # 🔥 4. Fill phần còn lại
    for key in ["street", "ward", "district"]:
        if result[key] is None and remaining:
            result[key] = remaining.pop(0)
    
    return pd.Series(result)

In [9]:
df[["street", "ward", "district", "city"]] = df["location_clean"].apply(parse_location)

df[["location_clean", "street", "ward", "district", "city"]].head(30)

,location_clean,street,ward,district,city
0,"đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ...",đường lỗ giáng 8,phường hòa xuân,quận cẩm lệ,đà nẵng
1,"đường cao đạt, phường 1, quận 5, tp hồ chí minh",đường cao đạt,phường 1,quận 5,tp hồ chí minh
2,"đường quốc lộ 13, thị trấn lai uyên, huyện bàu...",đường quốc lộ 13,thị trấn lai uyên,huyện bàu bàng,bình dương
3,"đường võ văn vân, xã vĩnh lộc b, huyện bình ch...",đường võ văn vân,xã vĩnh lộc b,huyện bình chánh,tp hồ chí minh
4,"thôn 2, xã suối rao, huyện châu đức, bà rịa - ...",thôn 2,xã suối rao,huyện châu đức,bà rịa - vũng tàu
5,"đường lý thái tổ, xã đạm bri, thành phố bảo lộ...",đường lý thái tổ,xã đạm bri,thành phố bảo lộc,lâm đồng
6,"đường số 1, phường trường thọ, quận thủ đức,...",đường số 1,phường trường thọ,quận thủ đức,tp hồ chí minh
7,"đường quốc lộ 20, thị trấn lộc thắng, huyện bả...",đường quốc lộ 20,thị trấn lộc thắng,huyện bảo lâm,lâm đồng
8,"đường 359, xã tân dương, huyện thuỷ nguyên, hả...",đường 359,xã tân dương,huyện thuỷ nguyên,hải phòng
9,"đường phó cơ điều, phường 12, quận 5, tp hồ ch...",đường phó cơ điều,phường 12,quận 5,tp hồ chí minh


In [10]:
test = "Thôn Lộc Châu 2, Xã Tân Nghĩa, Huyện Di Linh, Lâm Đồng"

print(parse_location(clean_location(test)))

street      thôn lộc châu 2
ward           xã tân nghĩa
district      huyện di linh
city               lâm đồng
dtype: object


In [11]:
df[df["city"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,None,None,None
50,Đường Phạm Văn Đồng,đường phạm văn đồng,None,None,None
116,Đường Đinh Đức Thiện,đường đinh đức thiện,None,None,None
165,Đường D7,đường d7,None,None,None
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,None,None,None
261,Đường Bùi Đình Túy,đường bùi đình túy,None,None,None
368,Hương An,hương an,None,None,None
414,D6,d6,None,None,None
679,Đường D7,đường d7,None,None,None
785,Duong go xoai,duong go xoai,None,None,None


In [12]:
df[df["district"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,None,None,None
50,Đường Phạm Văn Đồng,đường phạm văn đồng,None,None,None
116,Đường Đinh Đức Thiện,đường đinh đức thiện,None,None,None
165,Đường D7,đường d7,None,None,None
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,None,None,None
261,Đường Bùi Đình Túy,đường bùi đình túy,None,None,None
368,Hương An,hương an,None,None,None
414,D6,d6,None,None,None
679,Đường D7,đường d7,None,None,None
785,Duong go xoai,duong go xoai,None,None,None


In [13]:
df[df["ward"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,None,None,None
50,Đường Phạm Văn Đồng,đường phạm văn đồng,None,None,None
116,Đường Đinh Đức Thiện,đường đinh đức thiện,None,None,None
165,Đường D7,đường d7,None,None,None
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,None,None,None
261,Đường Bùi Đình Túy,đường bùi đình túy,None,None,None
368,Hương An,hương an,None,None,None
414,D6,d6,None,None,None
679,Đường D7,đường d7,None,None,None
785,Duong go xoai,duong go xoai,None,None,None


In [14]:
df[df["street"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
834,"200, Xã Bình Hiệp, Huyện Bình Sơn, Quảng Ngãi",None,xã bình hiệp,huyện bình sơn,quảng ngãi
1394,"301, Xã Tân Thạnh Đông, Huyện Củ Chi, Tp Hồ Ch...",None,xã tân thạnh đông,huyện củ chi,tp hồ chí minh
1693,"322, Xã Tân Phước, Huyện Đồng Phú, Bình Phước",None,xã tân phước,huyện đồng phú,bình phước
1719,"322, Xã Tân Phước, Huyện Đồng Phú, Bình Phước",None,xã tân phước,huyện đồng phú,bình phước
2120,"xã Duy Nghĩa, Xã Duy Nghĩa, Huyện Duy Xuyên, Q...",None,xã duy nghĩa,huyện duy xuyên,quảng nam
2327,"980, Phường Phú Hữu, Quận 9, Tp Hồ Chí Minh",None,phường phú hữu,quận 9,tp hồ chí minh
2526,"1, Xã An Linh, Huyện Phú Giáo, Bình Dương",None,xã an linh,huyện phú giáo,bình dương
2532,"943, Xã Vĩnh Thành, Huyện Châu Thành, An Giang",None,xã vĩnh thành,huyện châu thành,an giang
2791,"713, Thị trấn Đạ M'ri, Huyện Đạ Huoai, Lâm Đồng",None,thị trấn đạ m'ri,huyện đạ huoai,lâm đồng
3331,"105, Phường Tân Phú, Quận 9, Tp Hồ Chí Minh",None,phường tân phú,quận 9,tp hồ chí minh


In [15]:
parse_location("Đường Tỉnh lộ 15")

street      đường tỉnh lộ 15
ward                    None
district                None
city                    None
dtype: object

In [16]:
df.columns

Index(['title', 'price', 'area', 'location', 'description', 'Diện tích đất:',
       'Giá/m2:', 'Hướng cửa chính:', 'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:',
       'Loại hình đất:', 'Chiều ngang:', 'Chiều dài:', 'Số phòng ngủ:',
       'Số phòng vệ sinh:', 'Loại hình nhà ở:', 'Tình trạng nội thất:',
       'Diện tích sử dụng:', 'Tình trạng bất động sản:', 'Diện tích:',
       'Loại hình căn hộ:', 'Tổng số tầng:', 'Tên phân khu/Lô/Block/Tháp:',
       'Mã căn / Mã căn hộ:', 'Tầng số:', 'Hướng ban công:', 'Mã lô:',
       'Đặc điểm căn hộ:', 'Loại hình văn phòng:', 'location_clean', 'parts',
       'street', 'ward', 'district', 'city'],
      dtype='object')

In [17]:
# ✅ Drop cột trung gian
df = df.drop(columns=["location_clean", "parts"], errors="ignore")

# ✅ Save checkpoint Step 1
df.to_csv("chotot_step1_location.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved: chotot_step1_location.csv")
print(f"   Shape: {df.shape[0]:,} dòng | {df.shape[1]} cột")

✅ Saved: chotot_step1_location.csv
   Shape: 9,004 dòng | 33 cột


---

## 📦 Bước 2 — Xử lý cột Category

Load dữ liệu từ checkpoint `chotot_step1_location.csv` — đã có đầy đủ các cột location được parse ở Bước 1.

> **Pipeline:** Kiểm tra missing → Phân tích nguyên nhân → Fill `Unknown` → Verify → Save

In [18]:
df = pd.read_csv("chotot_step1_location.csv")
print(f"Load OK: {df.shape[0]:,} dòng | {df.shape[1]} cột")

Load OK: 9,004 dòng | 33 cột


## 🔍 Tổng quan Missing Data — Các cột Category

Kiểm tra toàn bộ các cột dạng categorical để xác định số lượng giá trị thiếu trước khi xử lý.

In [19]:
def check_category(df, col_name, top_n=10):
    print(f"=== COLUMN: {col_name} ===")
    
    # 1. Số category
    print("\n🔢 Number of unique values:")
    print(df[col_name].nunique())
    
    # 2. Số missing
    print("\n⚠️ Missing values:")
    print(df[col_name].isna().sum())
    
    # 3. Top value phổ biến
    print(f"\n📊 Top {top_n} values:")
    print(df[col_name].value_counts().head(top_n))
    
    # 4. List unique (optional)
    print("\n📋 Sample unique values:")
    print(df[col_name].dropna().unique()[:top_n])

In [20]:
cols = ["Hướng cửa chính:",'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:', 'Loại hình đất:', 'Loại hình nhà ở:', 'Tình trạng nội thất:', 'Tình trạng bất động sản:','Loại hình căn hộ:', 'Tên phân khu/Lô/Block/Tháp:','Hướng ban công:', 'Đặc điểm căn hộ:', 'Loại hình văn phòng:']

for col in cols:
    check_category(df, col)

=== COLUMN: Hướng cửa chính: ===

🔢 Number of unique values:
8

⚠️ Missing values:
0

📊 Top 10 values:
Hướng cửa chính:
Đông Nam    2123
Tây Nam     1181
Tây Bắc     1160
Đông Bắc    1034
Đông        1019
Nam          932
Bắc          825
Tây          730
Name: count, dtype: int64

📋 Sample unique values:
['Nam' 'Bắc' 'Đông Bắc' 'Tây Nam' 'Đông Nam' 'Đông' 'Tây' 'Tây Bắc']
=== COLUMN: Giấy tờ pháp lý: ===

🔢 Number of unique values:
3

⚠️ Missing values:
0

📊 Top 10 values:
Giấy tờ pháp lý:
Đã có sổ        8325
Đang chờ sổ      437
Giấy tờ khác     242
Name: count, dtype: int64

📋 Sample unique values:
['Đã có sổ' 'Đang chờ sổ' 'Giấy tờ khác']
=== COLUMN: Đặc điểm nhà/đất: ===

🔢 Number of unique values:
3

⚠️ Missing values:
0

📊 Top 10 values:
Đặc điểm nhà/đất:
Hẻm xe hơi    3850
Nở hậu        3009
Mặt tiền      2145
Name: count, dtype: int64

📋 Sample unique values:
['Mặt tiền' 'Nở hậu' 'Hẻm xe hơi']
=== COLUMN: Loại hình đất: ===

🔢 Number of unique values:
4

⚠️ Missing values:
0


### 📊 Kết quả kiểm tra Missing Data

| Cột | Số NaN | Ghi chú |
|-----|--------|---------|
| `Hướng cửa chính` | 0 | ✅ |
| `Giấy tờ pháp lý` | 0 | ✅ |
| `Đặc điểm nhà/đất` | 0 | ✅ |
| `Loại hình đất` | 0 | ✅ |
| `Loại hình nhà ở` | 1 | Khả năng đất chưa xây |
| `Tình trạng nội thất` | 1 | Khả năng đất chưa xây |
| `Tình trạng bất động sản` | 6 | Cần kiểm tra chi tiết |
| `Loại hình căn hộ` | 6 | Cần kiểm tra chi tiết |
| `Tên phân khu/Lô/Block/Tháp` | 35 | Không phải tất cả BĐS có phân khu |
| `Hướng ban công` | 47 | Không phải tất cả có ban công |
| `Đặc điểm căn hộ` | 69 | Không phải tất cả là căn hộ |
| `Loại hình văn phòng` | 291 | Không phải tất cả là văn phòng |

> **Nhận xét:** Các giá trị NaN ở trên phần lớn là **NA có nghĩa** — tức là thuộc tính đó không áp dụng cho loại bất động sản đó (ví dụ: đất chưa xây không có nội thất, không có hướng ban công). Sẽ kiểm tra chi tiết từng cột trước khi điền `Unknown`.

In [21]:
cols = [
    "Đặc điểm nhà/đất:",
    "Loại hình căn hộ:",
    "Tổng số tầng:",
    "Tên phân khu/Lô/Block/Tháp:",
    "Mã căn / Mã căn hộ:",
    "Tầng số:",
    "Hướng ban công:",
    "Mã lô:",
    "Đặc điểm căn hộ:",
    "Loại hình văn phòng:"
]

In [22]:
def check_na_with_related_cols(df, target_col, related_cols, n=20):
    # điều kiện NaN hoặc empty string
    condition = (
        df[target_col].isna() |
        (df[target_col].astype(str).str.strip() == "")
    )
    
    # chọn cột cần xem
    cols_to_show = ["title", "location", target_col] + related_cols
    
    result = df.loc[condition, cols_to_show]
    
    print(f"🔎 Total missing rows in '{target_col}': {len(result)}")
    
    return result.head(n)

---

## 🏠 Cột `Loại hình nhà ở`

Kiểm tra các dòng bị thiếu giá trị để xác định nguyên nhân NaN.

In [23]:
check_na_with_related_cols(df, "Loại hình nhà ở:", cols)

🔎 Total missing rows in 'Loại hình nhà ở:': 1


,title,location,Loại hình nhà ở:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df["Loại hình nhà ở:"] = (
    df["Loại hình nhà ở:"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace("", "Unknown")
)

**Nhận xét:** Dòng bị NaN có khả năng cao là **đất chưa xây nhà** — không có loại hình nhà ở nên bỏ trống là hợp lý. Giữ lại dòng này, điền `Unknown` để đánh dấu.

---

## 🛋️ Cột `Tình trạng nội thất`

Kiểm tra các dòng bị thiếu giá trị.

In [25]:
check_na_with_related_cols(df, "Tình trạng nội thất:", cols)

🔎 Total missing rows in 'Tình trạng nội thất:': 1


,title,location,Tình trạng nội thất:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df["Tình trạng nội thất:"] = (
    df["Tình trạng nội thất:"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace("", "Unknown")
)

**Nhận xét:** NaN ở cột này trùng với các dòng đất chưa xây — đất không có nội thất nên giá trị bỏ trống là hợp lý. Giữ lại, điền `Unknown`.

---

## 🏗️ Cột `Tình trạng bất động sản`

Kiểm tra các dòng bị thiếu giá trị.

In [27]:
check_na_with_related_cols(df, "Tình trạng bất động sản:", cols)

🔎 Total missing rows in 'Tình trạng bất động sản:': 6


,title,location,Tình trạng bất động sản:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
df["Tình trạng bất động sản:"] = (
    df["Tình trạng bất động sản:"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace("", "Unknown")
)

---

## 🏢 Cột `Loại hình căn hộ`

Kiểm tra các dòng bị thiếu giá trị và đánh giá từng trường hợp.

In [29]:
check_na_with_related_cols(df, "Loại hình căn hộ:", cols)

🔎 Total missing rows in 'Loại hình căn hộ:': 6


,title,location,Loại hình căn hộ:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Đánh giá từng dòng NaN:**

| # | Title | Kết luận | Xử lý |
|---|-------|----------|-------|
| 0 | Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân | Đất chưa xây nhà | Giữ lại |
| 1 | Nhà mặt tiền 1 trệt 2 lầu | Thiếu `Tổng số tầng`, `Mã căn hộ` → dữ liệu không đủ | **Loại bỏ** |
| 2 | Đất ngay TTHC Bầu Bàng | Đất chưa xây nhà | Giữ lại |
| 3 | Nhà đẹp 4x15m, Võ Văn Vân | Title có "Nhà" nhưng không có thông tin → dữ liệu thiếu | **Loại bỏ** |
| 4 | Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao | Đất chưa xây nhà | Giữ lại |
| 5 | 600m2 đất thổ cư SHR cách đường Lý Thái Tổ | Đất chưa xây nhà | Giữ lại |

**Kết luận:** Loại bỏ các dòng có `title` chứa từ "Nhà" nhưng lại thiếu `Loại hình căn hộ` → đây là dữ liệu bị lỗi, không có giá trị phân tích.

In [30]:
df = df[
    ~(
        df["Loại hình căn hộ:"].isna() & 
        df["title"].str.contains("Nhà", case=False, na=False)
    )
]

In [31]:
check_na_with_related_cols(df, "Loại hình căn hộ:", cols)

🔎 Total missing rows in 'Loại hình căn hộ:': 4


,title,location,Loại hình căn hộ:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
df["Loại hình căn hộ:"] = (
    df["Loại hình căn hộ:"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace("", "Unknown")
)

---

## 🏘️ Cột `Tên phân khu / Lô / Block / Tháp`

Kiểm tra missing và thực hiện **Feature Engineering** để chuẩn hóa cột này.

### 📐 Feature Engineering — Tên phân khu / Lô / Block / Tháp

Cột này là **mixed data** chứa nhiều loại thông tin khác nhau, cần được phân loại trước khi sử dụng.

---

#### Phân loại dữ liệu

| Nhóm | Mô tả | Ví dụ |
|------|-------|-------|
| `project` | Tên dự án / khu dân cư | `KHU DÂN CƯ AN PHÚ HƯNG`, `SÀI GÒN VILLAGE` |
| `block` | Ký hiệu block / lô | `A`, `B2`, `LÔ E`, `BLOCK B`, `A B` |
| `other` | Nhiễu / mô tả tự do | `NHÀ ĐẸP GẦN CÔNG VIÊN`, `12345` |

---

#### Pipeline xử lý: `Clean → Classify → Normalize → Assign`

**Bước 1 — Cleaning:**
Chuyển toàn bộ về chữ hoa, loại bỏ khoảng trắng dư.

**Bước 2 — Classification:**

- `project`: chuỗi chứa keyword `KHU`, `DỰ ÁN`, `VILLAGE`, `CITY`, `RESIDENCE`, `PARADISE`, `RIVERSIDE`, `URBAN`, `KDC`
- `block`: match regex `[A-Z]\d{0,2}` hoặc `(LÔ|BLOCK)\s*[A-Z0-9]+` hoặc `[A-Z](\s+[A-Z0-9]+)+`
- `other`: tất cả trường hợp còn lại

**Bước 3 — Normalize Block:**
Tách → bỏ trùng → sắp xếp → join bằng dấu phẩy

| Input | Output |
|-------|--------|
| `A B` | `A,B` |
| `B A` | `A,B` |
| `A A B` | `A,B` |

**Bước 4 — Tạo cột `ten_phan_khu_final`:**

| Loại | Giá trị cuối |
|------|-------------|
| `project` | Giữ nguyên tên dự án |
| `block` | Chuỗi block đã normalize |
| `other` | `"Không thuộc project/block"` |

---

> **Lưu ý:** Cột này không phải categorical thuần — chứa entity, code và text noise lẫn lộn. Rule-based phù hợp hơn ML trong trường hợp này do dữ liệu nhiễu cao.

In [33]:
check_na_with_related_cols(df, "Tên phân khu/Lô/Block/Tháp:", cols)

🔎 Total missing rows in 'Tên phân khu/Lô/Block/Tháp:': 33


,title,location,Tên phân khu/Lô/Block/Tháp:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,"Bán Nhà 1 Trệt 2 Lầu, HXH, P.HBP, 54m2, Giá 4....","Đường số 8, Phường Hiệp Bình Phước, Quận Thủ Đ...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Bán Lô Đất 3300M2 1Tr4/M2Lý Thường Kiệt Tp Bảo...,"Đường Lý Thường Kiệt, Phường Lộc Phát, Thành p...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
# Clear Data
def clean_text(x):
    if pd.isna(x):
        return x
    x = x.upper().strip()
    x = re.sub(r"\s+", " ", x)
    return x

df["ten_phan_khu_clean"] = df["Tên phân khu/Lô/Block/Tháp:"].apply(clean_text)


In [35]:
df["ten_phan_khu_clean"].value_counts()

ten_phan_khu_clean
A                                1066
B                                 728
THÔN BÌNH KHÁNH                   275
1                                 239
C                                 222
                                 ... 
CAM LÂM, NHA TRANG, KHÁNH HÒA       1
LÔ B                                1
PHÂN KHU 2                          1
749                                 1
L                                   1
Name: count, Length: 244, dtype: int64

In [36]:
for val in df["ten_phan_khu_clean"].dropna().unique():
    print(val)

DỰ ÁN TÍN HƯNG
B
KHU DÂN CƯ AN PHÚ HƯNG
BỆNH VIỆN XUYÊN Á TÂY NINH
A
C
PHƯỚC BÌNH
ABCD
AN BÌNH
BAU BÀNG
KHU ĐƯỜNG BÀN CỜ PHÚ THẠNH-PHÚ THỌ HÒA
L43, L44, L45
RIGEL
D1
115
ĐẤT TÂN HIỆP
KBD
THÔN BÌNH KHÁNH
KDC
DỊCH VỤ 6.9
D
SÀI GÒN VILLAGE
BABYLON
NHÀ DOI DIEN CONG VIEN,RAT MAT ME.
MP1
B3
KHU ĐÔ THỊ TRƯỜNG AN
A,B
DCH
NHÀ PHỐ LIỀN KỀ
E3
KHU HOÀNG HOA THÁM
A B
KHU DÂN CU
1
HOMELAND PARADISE VILLA
2
TOM 77
H
KDCD
NAM CẨM LỆ
MANHATTAN
C4,C5,A4
N16
H20
HẺM 481 ĐƯỜNG TÂN KỲ TÂN QUÝ
HANEL SÀI ĐỒNG LONG BIÊN
03
749
E
NGỌC ĐỊNH FARM
T5
LÔ E
LIỀN KHU DÂN CƯ THỊNH VƯỢNG
CHÂU THỚI
ĐẠI THÀNH NGHI KIM
NGÕ 37 ĐẠI ĐỒNG
DỰ ÁN KHU NGHĨ DƯỠNG BÃI DÀI PHÚ QUỐC
00
A8
BLOCK B
VĨNH ĐIỀM THƯỢNG
ẤP 1 SÔNG TRẦU THỊ TRẤN TRẢNG BOM
KHU DÂN CƯ
KHU DÂN CƯ THẠNH MỸ LỢI DRAGON
BÌNH KHÁNH 2
S2.02
ĐẢO THỊNH VƯỢNG TAM ĐA
ĐƯỜNG 4A
TA15
THE MANHATTAN GLORY
BÌNH NGUYÊN
A1
KHU DAN CƯ CAO CÂP
6-MAY
MẶT TIỀN 833
G50
HẺM TRỊNH ĐÌNH TRỌNG
L K J H F G
CHÙA ĐỨC VIÊN
B2.11
965
NHÀ HẺM
NAM LONG
01
KHU DÂN CƯ NINH GIANG CÁT LÁI
CT1
MẶT

In [37]:
# Classify Data
def classify_type(x):
    if pd.isna(x):
        return "other"
    
    if any(k in x for k in [
        "KHU", "DỰ ÁN", "VILLAGE", "CITY", "RESIDENCE", "RESIDENCES",
        "PARADISE", "RIVERSIDE", "URBAN", "KDC"
    ]):
        return "project"
    
    if re.fullmatch(r"[A-Z]\d{0,2}", x):
        return "block"
    
    if re.fullmatch(r"(LÔ|BLOCK)\s*[A-Z0-9]+", x):
        return "block"
    
    if re.fullmatch(r"[A-Z](\s+[A-Z0-9]+)+", x):
        return "block"
    
    return "other"

In [38]:
df["phan_loai"] = df["ten_phan_khu_clean"].apply(classify_type)

In [39]:
# NORMALIZE BLOCK
def normalize_block(x):
    if pd.isna(x):
        return x
    parts = re.split(r"\s+", x)
    parts = sorted(set(parts))
    return ",".join(parts)

In [40]:
def normalize_block(x):
    parts = re.split(r"[,\s]+", x)
    parts = sorted(set(parts))
    return ",".join(parts)

df.loc[df["phan_loai"] == "block", "ten_phan_khu_final"] = (
    df.loc[df["phan_loai"] == "block", "ten_phan_khu_clean"]
    .apply(normalize_block)
)

In [41]:
# FINAL COLUMN
df["ten_phan_khu_final"] = None

# project giữ nguyên
df.loc[df["phan_loai"] == "project", "ten_phan_khu_final"] = df["ten_phan_khu_clean"]

# block normalize
df.loc[df["phan_loai"] == "block", "ten_phan_khu_final"] = (
    df.loc[df["phan_loai"] == "block", "ten_phan_khu_clean"]
    .apply(normalize_block)
)

# other
df.loc[df["phan_loai"] == "other", "ten_phan_khu_final"] = "Không thuộc project/block"

In [42]:
df[["Tên phân khu/Lô/Block/Tháp:","ten_phan_khu_clean", "phan_loai", "ten_phan_khu_final"]].head(40)

,Tên phân khu/Lô/Block/Tháp:,ten_phan_khu_clean,phan_loai,ten_phan_khu_final
0,NaN,NaN,other,Không thuộc project/block
2,NaN,NaN,other,Không thuộc project/block
4,NaN,NaN,other,Không thuộc project/block
5,NaN,NaN,other,Không thuộc project/block
6,NaN,NaN,other,Không thuộc project/block
7,NaN,NaN,other,Không thuộc project/block
8,NaN,NaN,other,Không thuộc project/block
9,NaN,NaN,other,Không thuộc project/block
10,NaN,NaN,other,Không thuộc project/block
11,NaN,NaN,other,Không thuộc project/block


---

## 🧭 Cột `Hướng ban công`

Kiểm tra missing và điền `Unknown` cho các giá trị thiếu.

In [43]:
check_na_with_related_cols(df, "Hướng ban công:", cols)

🔎 Total missing rows in 'Hướng ban công:': 45


,title,location,Hướng ban công:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,"Bán Nhà 1 Trệt 2 Lầu, HXH, P.HBP, 54m2, Giá 4....","Đường số 8, Phường Hiệp Bình Phước, Quận Thủ Đ...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Bán Lô Đất 3300M2 1Tr4/M2Lý Thường Kiệt Tp Bảo...,"Đường Lý Thường Kiệt, Phường Lộc Phát, Thành p...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
df["Hướng ban công:"] = (
    df["Hướng ban công:"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace("", "Unknown")
)

In [45]:
df["Hướng ban công:"].value_counts()

Hướng ban công:
Tây Bắc     2388
Đông Nam    2303
Đông Bắc    1686
Bắc          655
Nam          629
Tây Nam      498
Đông         482
Tây          316
Unknown       45
Name: count, dtype: int64

**Nhận xét:** Giá trị `Unknown` ở cột này phản ánh 2 trường hợp:
- Căn hộ / nhà **không có ban công**
- **Đất chưa xây** nên chưa có thông tin hướng

---

## 🏠 Cột `Đặc điểm căn hộ`

Kiểm tra missing và điền `Unknown` cho các giá trị thiếu.

In [46]:
check_na_with_related_cols(df, "Đặc điểm căn hộ:", cols)

🔎 Total missing rows in 'Đặc điểm căn hộ:': 67


,title,location,Đặc điểm căn hộ:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
10,"Bán Nhà 1 Trệt 2 Lầu, HXH, P.HBP, 54m2, Giá 4....","Đường số 8, Phường Hiệp Bình Phước, Quận Thủ Đ...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
11,Bán Lô Đất 3300M2 1Tr4/M2Lý Thường Kiệt Tp Bảo...,"Đường Lý Thường Kiệt, Phường Lộc Phát, Thành p...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN


In [47]:
df["Đặc điểm căn hộ:"] = (
    df["Đặc điểm căn hộ:"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace("", "Unknown")
)

In [48]:
df["Đặc điểm căn hộ:"].value_counts()

Đặc điểm căn hộ:
Căn góc    8935
Unknown      67
Name: count, dtype: int64

**Nhận xét:** Giá trị `Unknown` ở cột này phản ánh 2 trường hợp:
- Bất động sản **không phải căn hộ** (nhà phố, đất nền...)
- Thông tin **chưa được cập nhật** bởi người đăng

---

## 🏢 Cột `Loại hình văn phòng`

Kiểm tra missing và điền `Unknown` cho các giá trị thiếu. Đây là cột có số NaN **lớn nhất** (291 dòng) do phần lớn bất động sản trong dataset không phải văn phòng.

In [49]:
check_na_with_related_cols(df, "Loại hình văn phòng:", cols)

🔎 Total missing rows in 'Loại hình văn phòng:': 289


,title,location,Loại hình văn phòng:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,Unknown,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
10,"Bán Nhà 1 Trệt 2 Lầu, HXH, P.HBP, 54m2, Giá 4....","Đường số 8, Phường Hiệp Bình Phước, Quận Thủ Đ...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN
11,Bán Lô Đất 3300M2 1Tr4/M2Lý Thường Kiệt Tp Bảo...,"Đường Lý Thường Kiệt, Phường Lộc Phát, Thành p...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,Unknown,NaN,Unknown,NaN


In [50]:
df["Loại hình văn phòng:"] = (
    df["Loại hình văn phòng:"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace("", "Unknown")
)

In [51]:
df["Loại hình văn phòng:"].value_counts()

Loại hình văn phòng:
Mặt bằng kinh doanh    6078
Shophouse              2454
Unknown                 289
Văn phòng               181
Name: count, dtype: int64

**Nhận xét:** Số lượng NaN lớn (291 dòng) là bình thường — chỉ các tin đăng văn phòng mới có giá trị, phần còn lại để trống. Điền `Unknown` để thống nhất format.

In [55]:
# ✅ Kiểm tra NaN còn lại
category_cols = [
    "Hướng cửa chính:", "Giấy tờ pháp lý:", "Đặc điểm nhà/đất:",
    "Loại hình đất:", "Loại hình nhà ở:", "Tình trạng nội thất:",
    "Tình trạng bất động sản:", "Loại hình căn hộ:",
    "Tên phân khu/Lô/Block/Tháp:", "Hướng ban công:",
    "Đặc điểm căn hộ:", "Loại hình văn phòng:",
    # derived columns từ Tên phân khu
    "ten_phan_khu_clean", "phan_loai", "ten_phan_khu_final",
]

print(f"{'Cột':<40} {'NaN còn lại':>12} {'Status':>10}")
print("-" * 65)
all_clean = True
for col in category_cols:
    if col not in df.columns:
        print(f"{col:<40} {'KHÔNG TỒN TẠI':>12} {'⚠️':>10}")
        continue
    n = df[col].isna().sum() + (df[col].astype(str).str.strip() == "").sum()
    status = "✅ OK" if n == 0 else "❌ CÒN NaN"
    if n > 0: all_clean = False
    print(f"{col:<40} {n:>12} {status:>10}")
print("-" * 65)
print("\n🎉 Tất cả OK!" if all_clean else "\n⚠️ Vẫn còn NaN!")

Cột                                       NaN còn lại     Status
-----------------------------------------------------------------
Hướng cửa chính:                                    0       ✅ OK
Giấy tờ pháp lý:                                    0       ✅ OK
Đặc điểm nhà/đất:                                   0       ✅ OK
Loại hình đất:                                      0       ✅ OK
Loại hình nhà ở:                                    0       ✅ OK
Tình trạng nội thất:                                0       ✅ OK
Tình trạng bất động sản:                            0       ✅ OK
Loại hình căn hộ:                                   0       ✅ OK
Tên phân khu/Lô/Block/Tháp:                        33  ❌ CÒN NaN
Hướng ban công:                                     0       ✅ OK
Đặc điểm căn hộ:                                    0       ✅ OK
Loại hình văn phòng:                                0       ✅ OK
ten_phan_khu_clean                                 33  ❌ CÒN NaN
phan_loai               

In [56]:
df.to_csv("chotot_step2_category.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved: chotot_step2_category.csv")
print(f"   Shape: {df.shape[0]:,} dòng | {df.shape[1]} cột")

✅ Saved: chotot_step2_category.csv
   Shape: 9,002 dòng | 36 cột
